# LLM Monetary Policy Classification Demo

Demonstrates fiscal and monetary policy classification prompts with 4 variants each: simple, with_definitions, chain_of_thought, and few_shot.

In [11]:
# Setup
from dotenv import load_dotenv
load_dotenv('../../.env')
import os, sys
sys.path.insert(0, '../../libs')
sys.path.insert(0, '../../src/Traction/prompts')
sys.path.insert(0, '../../src/Traction')
import config
from llm_factory_openai import LLMAgent
# Use _process_batch_async to process batch_messages
import asyncio
from llm_batch_process_utils import _process_batch_async
from llm_factory_openai import BatchAsyncLLMAgent
from llm_batch_process_utils import _merge_ids_with_responses

from prompt_utils import load_prompt, format_messages
from schema import PROMPT_REGISTRY
from pathlib import Path
import pandas as pd
from llm_batch_process_utils import _build_batch_messages_from_df_flexible
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

#### Set up llm

In [12]:
# Initialize LLM Agent
llm_agent = LLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5-nano',
    temperature=1
)

#### Load prompts templates

In [13]:
# Group prompts by category
prompts = {'monetary_stance': [], 'monetary_agreement': [], 'fiscal_stance': [], 'fiscal_agreement': []}
for key, entry in PROMPT_REGISTRY.items():
    fname = entry["prompt_file"]
    for category in prompts.keys():
        if fname.startswith(category):
            prompts[category].append(key)
            break

print(f"Loaded {sum(len(v) for v in prompts.values())} prompts across {len(prompts)} categories")

Loaded 16 prompts across 4 categories


In [14]:
prompts

{'monetary_stance': ['monetary_stance_simple',
  'monetary_stance_with_definitions',
  'monetary_stance_few_shot',
  'monetary_stance_chain_of_thought'],
 'monetary_agreement': ['monetary_agreement_simple',
  'monetary_agreement_with_definitions',
  'monetary_agreement_few_shot',
  'monetary_agreement_chain_of_thought'],
 'fiscal_stance': ['fiscal_stance_simple',
  'fiscal_stance_with_definitions',
  'fiscal_stance_few_shot',
  'fiscal_stance_chain_of_thought'],
 'fiscal_agreement': ['fiscal_agreement_simple',
  'fiscal_agreement_with_definitions',
  'fiscal_agreement_few_shot',
  'fiscal_agreement_chain_of_thought']}

## Load data

In [15]:
data_dir = Path(config.data_dir / 'Finetuning_data')
data_dir.exists()

True

In [16]:
df = pd.read_excel(data_dir / 'labelled_monetary_v6.xlsx')
#df = pd.read_excel('/data/home/xiong/data/Fund/CSR/Tractions/Finetuning_data/labelled_fiscal_v2.xlsx')
df_agree = df[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]
df_stance = pd.DataFrame()
for tp in ['staff', 'buff']:
    df_temp = df[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_current'%tp, '%s_stance_future'%tp]]
    df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
    df_temp['type'] = tp
    df_stance = pd.concat([df_stance, df_temp], ignore_index=True)
    
# Remove the 'index' column and reset index to create a new 'index' column
df_stance = df_stance.drop(columns=['index']).reset_index(names='index')

#### Work on agreement

In [17]:
for col in ['agreement_general', 'disagreement_areas']:
    print(df_agree[col].value_counts(normalize=True))

agreement_general
mostly agree           0.702422
disagreement exists    0.217993
irrelevant             0.079585
Name: proportion, dtype: float64
disagreement_areas
Future Policy Direction                                                      0.484848
Current Policy Stance                                                        0.121212
Monetary Policy Operations                                                   0.075758
Central Bank Communication                                                   0.045455
Monetary Policy Framework                                                    0.045455
Economic Assessment                                                          0.045455
Current Policy Stance; Future Policy Direction                               0.030303
Economic Assessment; Future Policy Direction                                 0.030303
Exchange Rate Policy                                                         0.030303
Monetary Policy Framework; Current Policy Stance; Future Pol

In [18]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = 'monetary_agreement_chain_of_thought'
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_file = registry_entry["prompt_file"]
prompt_dir = Path('../../src/Traction/prompts')
# Load prompt template
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [19]:
PROMPT_REGISTRY

{'topic_classification': {'prompt_file': 'topic_classification.md',
  'response_model': schema.TopicResponse},
 'monetary_stance_simple': {'prompt_file': 'monetary_stance_simple.md',
  'response_model': schema.MonetaryStanceResponse},
 'monetary_stance_with_definitions': {'prompt_file': 'monetary_stance_with_definitions.md',
  'response_model': schema.MonetaryStanceResponse},
 'monetary_stance_few_shot': {'prompt_file': 'monetary_stance_few_shot.md',
  'response_model': schema.MonetaryStanceResponse},
 'monetary_stance_chain_of_thought': {'prompt_file': 'monetary_stance_chain_of_thought.md',
  'response_model': schema.MonetaryStanceChainOfThoughtResponse},
 'monetary_agreement_simple': {'prompt_file': 'monetary_agreement_simple.md',
  'response_model': schema.MonetaryAgreementResponse},
 'monetary_agreement_with_definitions': {'prompt_file': 'monetary_agreement_with_definitions.md',
  'response_model': schema.MonetaryAgreementResponse},
 'monetary_agreement_few_shot': {'prompt_file': '

In [20]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'STAFF_TEXT': 'staff',
    'AUTHORITY_TEXT': 'buff'
}

In [21]:
# combine both traing and testing data
# sample_df = pd.concat([df_train_agree, df_test_agree])
sample_df = df_agree

In [22]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    sample_df,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=10000
)

In [23]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:5500]}...")


Generated 289 batch messages
Sample IDs: ['0', '1', '2', '3', '4', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '278', '279', '281', '282', '283', '284', '285', '287', '288', '289', '290', '291', '292', '293', '294', '295', '

In [24]:
response_model.schema_json()


'{"properties": {"agreement": {"description": "Level of agreement between IMF staff and country authorities", "enum": ["irrelevant", "disagreement exists", "mostly agree"], "title": "Agreement", "type": "string"}, "disagreement_areas": {"description": "List of disagreement areas or empty string", "title": "Disagreement Areas", "type": "string"}, "reason": {"description": "Reasoning for the classification", "title": "Reason", "type": "string"}}, "required": ["agreement", "disagreement_areas", "reason"], "title": "MonetaryAgreementChainOfThoughtResponse", "type": "object"}'

In [25]:
try:
    response = llm_agent.get_response_content(batch_messages[0], reasoning_effort='low', response_format=response_model)
    print(response)
except Exception as e:
    print(f"Error: {e}")

agreement='disagreement exists' disagreement_areas='current policy stance; future policy direction; monetary policy framework' reason='IMF staff advocate prudent policy, readiness to raise rates, and structural measures (removing lending rate cap, lender of last resort). Authorities describe a tactic of gradual tightening with unchanged policy rate, emphasize modernization of framework and autonomy, and do not indicate readiness to raise rates or remove rate caps. The two texts thus show conflicting views on stance and direction of monetary policy and on policy framework choices.'


In [26]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-4o-2024-08-06',
    temperature=0
)

In [27]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=5000,
        #reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 289/289 [02:14<00:00,  2.14msg/s]


In [28]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")
print(merged[0])

Processed 289 responses
{'agreement': 'mostly agree', 'disagreement_areas': '', 'reason': "Both the IMF staff and the Tunisian authorities discuss aspects of monetary policy, and there is a general alignment in their views. The IMF staff appreciates the move towards positive real interest rates and the authorities' readiness to raise rates if inflationary pressures increase. The authorities also express vigilance towards inflationary pressures and have kept the policy rate unchanged due to weak economic activity, which aligns with a prudent approach. Additionally, both parties are focused on modernizing and strengthening the monetary policy framework, with the authorities working on improvements in cooperation with the Banque de France and benefiting from Fund technical assistance. There is no explicit disagreement expressed by the authorities on the issues discussed.", 'id': '0'}


In [29]:
# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

,agreement_llm,disagreement_areas_llm,reason_llm,id
0,mostly agree,,Both the IMF staff and the Tunisian authoritie...,0
1,mostly agree,,Both the IMF staff and the Ghanaian authoritie...,1
2,disagreement exists,"current policy stance, central bank communication",The IMF staff suggests that the BoJ needs to b...,2
3,disagreement exists,"current policy stance, future policy direction",The IMF staff recommends a fully flexible exch...,3
4,disagreement exists,current policy stance,The IMF staff suggests that the BoU should rem...,4
...,...,...,...,...
284,mostly agree,,Both the IMF staff and the Nepalese authoritie...,434
285,mostly agree,,Both the IMF staff and the Armenian authoritie...,435
286,mostly agree,,Both the IMF staff and the Colombian authoriti...,436
287,mostly agree,,Both the IMF staff and the Uruguayan authoriti...,437


In [30]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(sample_df, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=['agreement_general', 'agreement_llm'])
print(len(df_merged))
df_merged.head()

289


,agreement_llm,disagreement_areas_llm,reason_llm,id,index,Print ISBN,staff,buff,country,year,agreement_general,disagreement_areas
0,mostly agree,,Both the IMF staff and the Tunisian authoritie...,0,0,9781513529943,60. Monetary policy should remain prudent. Sta...,The gradual monetary policy tightening and dec...,Tunisia,2015,mostly agree,NaN
1,mostly agree,,Both the IMF staff and the Ghanaian authoritie...,1,1,9781484317259,56. Monetary policy has a key role to play in ...,"Continued tight monetary policy, exchange rate...",Ghana,2017,mostly agree,NaN
2,disagreement exists,"current policy stance, central bank communication",The IMF staff suggests that the BoJ needs to b...,2,2,9781513506180,49. More explicit monetary guidance would enha...,9. The quantitative and qualitative monetary e...,Japan,2015,mostly agree,NaN
3,disagreement exists,"current policy stance, future policy direction",The IMF staff recommends a fully flexible exch...,3,3,9781513579863,33. The exchange rate should be made fully fle...,Despite the efforts to address inflationary pr...,Republic of Belarus,2015,mostly agree,NaN
4,disagreement exists,current policy stance,The IMF staff suggests that the BoU should rem...,4,4,9781484309322,43. The BoU’s inflation targeting framework ha...,"Inflation edged up, mainly reflecting the effe...",Uganda,2017,mostly agree,NaN


In [31]:
# query v6:
print("accuracy Score:")
print(accuracy_score(df_merged['agreement_general'], 
               df_merged['agreement_llm']))

print("f1 Score:")      
print(f1_score(df_merged['agreement_general'], 
         df_merged['agreement_llm'], 
         average='weighted'))

print("Confusion Matrix:")
confusion_matrix(df_merged['agreement_general'], 
                 df_merged['agreement_llm'], 
                 labels=['disagreement exists', 
                         'mostly agree', 
                         'irrelevant'])

accuracy Score:
0.71280276816609
f1 Score:
0.6979667684706559
Confusion Matrix:


array([[ 38,  25,   0],
       [ 37, 166,   0],
       [  6,  15,   2]])

#### Try Stance prediction

In [32]:
# Demo: Build batch messages for monetary stance using flexible function
prompt_key = 'monetary_stance_chain_of_thought'
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_dir = Path('../../src/Traction/prompts')
prompt_file = registry_entry["prompt_file"]
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [33]:
prompt_template

{'system': 'You are an experienced macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of {TYPE}, complete the following two tasks.\n\nFirst, classify the country\'s recent or current monetary policy stance as described in the text into "restrictive"/"neutral"/"accommodative"/"unclear"/"irrelevant"; if it discusses monetary policy but the specific stance is not clear, assign "unclear"; if it does not discuss monetary policy, assign "irrelevant".\n\nSecond, classify the {TYPE_POSSESSIVE} {VERB} near-term (next year) direction of change in monetary policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". {EXPLANATION} If the text indicates a leaning towards tightening/loosening but without a full commitment, assign "tightening bias"/"loosening bias". \n\nIf the text discusses monetary policy stance but the direction of change is not clear, ass

In [34]:
df_stance.head()

,index,Print ISBN,country,year,text,stance_current,stance_future,type
0,0,9781513529943,Tunisia,2015,60. Monetary policy should remain prudent. Sta...,restrictive,tightening bias,staff
1,1,9781484317259,Ghana,2017,56. Monetary policy has a key role to play in ...,unclear,tightening bias,staff
2,2,9781513506180,Japan,2015,49. More explicit monetary guidance would enha...,accommodative,loosening bias,staff
3,3,9781513579863,Republic of Belarus,2015,33. The exchange rate should be made fully fle...,unclear,tightening,staff
4,4,9781484309322,Uganda,2017,43. The BoU’s inflation targeting framework ha...,unclear,no change,staff


In [36]:
for col in ['stance_current', 'stance_future']:
    print(df_stance[col].value_counts(normalize=True))
    print('\n')

stance_current
accommodative    0.370242
unclear          0.365052
restrictive      0.209343
irrelevant       0.038062
neutral          0.017301
Name: proportion, dtype: float64


stance_future
no change          0.467128
tightening bias    0.167820
tightening         0.131488
loosening bias     0.093426
unclear            0.058824
loosening          0.044983
irrelevant         0.036332
Name: proportion, dtype: float64




In [37]:
from prompt_examples import (
    AUTHOR_TYPE_MAPPING,
    AUTHOR_TYPE_POSSESSIVE_MAPPING,
    AUTHOR_VERB_MAPPING,
    MONETARY_STANCE_EXAMPLES,
    MONETARY_STANCE_EXPLANATIONS
)

In [40]:
type_dict1 = AUTHOR_TYPE_MAPPING
type_dict2 = AUTHOR_TYPE_POSSESSIVE_MAPPING
verb_dict = AUTHOR_VERB_MAPPING
example_dict = MONETARY_STANCE_EXAMPLES
explanation_dict = MONETARY_STANCE_EXPLANATIONS

# Add examples and explanation columns based on type
df_stance['examples'] = df_stance['type'].map(example_dict)
df_stance['explanation'] = df_stance['type'].map(explanation_dict)
df_stance['author_type'] = df_stance['type'].map(type_dict1)
df_stance['type_possessive'] = df_stance['type'].map(type_dict2)
df_stance['verb'] = df_stance['type'].map(verb_dict)

In [41]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'TEXT': 'text',
    'TYPE': 'author_type',
    'TYPE_POSSESSIVE': 'type_possessive',
    'VERB': 'verb',
    'EXAMPLES': 'examples',
    'EXPLANATION': 'explanation'
}

In [42]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    df_stance,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=8000
)

In [43]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:3000]}...")
""

Generated 578 batch messages
Sample IDs: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152

''

In [44]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-4o-2024-08-06',
    temperature=0
)

In [45]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=5000,
        #reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 578/578 [04:09<00:00,  2.32msg/s]


In [49]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")

# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

Processed 578 responses


,stance_current_llm,stance_future_llm,reason_llm,id
0,neutral,tightening bias,The text indicates that the monetary policy sh...,0
1,accommodative,tightening bias,The text discusses the monetary policy stance ...,1
2,accommodative,loosening bias,The text discusses the Bank of Japan's (BoJ) m...,2
3,accommodative,tightening,The text suggests that the current monetary po...,3
4,neutral,no change,The text indicates that the Bank of Uganda (Bo...,4
...,...,...,...,...
573,restrictive,no change,The text indicates that Nepal's current moneta...,573
574,accommodative,no change,The text indicates that the Central Bank of Ar...,574
575,accommodative,no change,The text describes Colombia's monetary policy ...,575
576,restrictive,no change,The text explicitly states that the Central Ba...,576


In [50]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(df_stance, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=[ 'stance_current','stance_future'])
print(len(df_merged))
df_merged.head()

578


,stance_current_llm,stance_future_llm,reason_llm,id,index,Print ISBN,country,year,text,stance_current,stance_future,type,examples,explanation,author_type,type_possessive,verb
0,neutral,tightening bias,The text indicates that the monetary policy sh...,0,0,9781513529943,Tunisia,2015,60. Monetary policy should remain prudent. Sta...,restrictive,tightening bias,staff,Example 1: Country: Guyana; Year: 2017; Text: ...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
1,accommodative,tightening bias,The text discusses the monetary policy stance ...,1,1,9781484317259,Ghana,2017,56. Monetary policy has a key role to play in ...,unclear,tightening bias,staff,Example 1: Country: Guyana; Year: 2017; Text: ...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
2,accommodative,loosening bias,The text discusses the Bank of Japan's (BoJ) m...,2,2,9781513506180,Japan,2015,49. More explicit monetary guidance would enha...,accommodative,loosening bias,staff,Example 1: Country: Guyana; Year: 2017; Text: ...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
3,accommodative,tightening,The text suggests that the current monetary po...,3,3,9781513579863,Republic of Belarus,2015,33. The exchange rate should be made fully fle...,unclear,tightening,staff,Example 1: Country: Guyana; Year: 2017; Text: ...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
4,neutral,no change,The text indicates that the Bank of Uganda (Bo...,4,4,9781484309322,Uganda,2017,43. The BoU’s inflation targeting framework ha...,unclear,no change,staff,Example 1: Country: Guyana; Year: 2017; Text: ...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended


In [51]:
df_merged['stance_current_llm'].value_counts()

stance_current_llm
accommodative    259
restrictive      147
neutral          109
irrelevant        43
unclear           20
Name: count, dtype: int64

In [52]:
df_merged['stance_current'].value_counts()

stance_current
accommodative    214
unclear          211
restrictive      121
irrelevant        22
neutral           10
Name: count, dtype: int64

In [ ]:
# query v1:
print("=== STANCE CURRENT METRICS ===")
print("Stance Current - Accuracy:", 
      accuracy_score(df_merged['stance_current'], df_merged['stance_current_llm']))
print("Stance Current - F1 Score:", 
      f1_score(df_merged['stance_current'], df_merged['stance_current_llm'], average='weighted'))

print("\n=== STANCE FUTURE METRICS ===")
print("Stance Future - Accuracy:", 
      accuracy_score(df_merged['stance_future'], df_merged['stance_future_llm']))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_future'], df_merged['stance_future_llm'], average='weighted'))

print("\n=== MERGING ONLY UNCLEAR/IRRELEVANT (KEEPING BIAS CATEGORIES) ===")
print("Stance Current - Accuracy:", 
      accuracy_score(df_merged['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
                     df_merged['stance_current_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x)))
print("Stance Current - F1 Score:", 
      f1_score(df_merged['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
               df_merged['stance_current_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'))

print("\nStance Future - Accuracy:", 
      accuracy_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
                     df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x)))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
               df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'))

print("\n=== MERGING UNCLEAR/IRRELEVANT AND BIAS CATEGORIES ===")
print("Stance Current - Accuracy:", 
      accuracy_score(df_merged['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), 
                     df_merged['stance_current_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', ''))))
print("Stance Current - F1 Score:", 
      f1_score(df_merged['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), 
               df_merged['stance_current_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), average='weighted'))

print("\nStance Future - Accuracy:", 
      accuracy_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), 
                     df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', ''))))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), 
               df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), average='weighted'))


=== STANCE CURRENT METRICS ===
Stance Current - Accuracy: 0.6072664359861591
Stance Current - F1 Score: 0.5577040248145986

=== STANCE FUTURE METRICS ===
Stance Future - Accuracy: 0.6851211072664359
Stance Future - F1 Score: 0.6734623976929771

=== MERGING ONLY UNCLEAR/IRRELEVANT (KEEPING BIAS CATEGORIES) ===
Stance Current - Accuracy: 0.6505190311418685
Stance Current - F1 Score: 0.6463174625629844

Stance Future - Accuracy: 0.7214532871972318
Stance Future - F1 Score: 0.7196721539740062

=== MERGING UNCLEAR/IRRELEVANT AND BIAS CATEGORIES ===
Stance Current - Accuracy: 0.6505190311418685
Stance Current - F1 Score: 0.6463174625629844

Stance Future - Accuracy: 0.7993079584775087
Stance Future - F1 Score: 0.7988613452581944
